# Clasificación: Comparación de Modelos

### Imports y Configuración

In [ ]:
import sys
sys.path.append("/workspace/src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from EstiloGraficos import aplicar_estilo

# Misma paleta y estilo que el dashboard (components/plotly_theme.py)
aplicar_estilo()

# Textos largos completos en las tablas (criterio de selección, hiperparámetros)
pd.set_option("display.max_colwidth", None)

from Modelado import (
    SEED, UMBRAL, RECALL_MINIMO, USAR_GPU, GPU_DISPOSITIVOS, PREDICTORES,
    NOMBRES_CLF, NOMBRES_FASE2, CLAVE_BENCHMARK, ESTRATEGIAS,
    cargar_split, separar_x_y, crear_cv, resumen_dispositivos,
    cargar_ganador_fase1, crear_benchmark, cargar_cv_fase1, formatear_tabla_cv, elegir_mejor,
    evaluar_test_modelos, graficar_curvas_test, intervalos_bootstrap, graficar_intervalos_bootstrap,
    formatear_hiperparametros, probabilidades_oof, elegir_umbrales, graficar_umbral, metricas_con_umbral, graficar_matrices,
    guardar_final, cargar_modelo_final,
)

In [ ]:
resumen_dispositivos().style.set_caption(f"USAR_GPU = {USAR_GPU}")

### Datos: Split y Folds

In [ ]:
df = pd.read_csv('/workspace/data/raw/screen_time_mental_health.csv')

df_train, df_test = cargar_split(df)
X_train, y_train = separar_x_y(df_train)
X_test, y_test = separar_x_y(df_test)
cv = crear_cv(SEED)

pd.DataFrame({
    "Filas": [len(X_train), len(X_test)],
    "Predictores": [X_train.shape[1], X_test.shape[1]],
    "Deprimidos": [int(y_train.sum()), int(y_test.sum())],
    "Prevalencia": [round(y_train.mean(), 3), round(y_test.mean(), 3)],
}, index=pd.Index(["Train", "Test"], name="Conjunto"))

## Comparación de Modelos

Se carga el Pipeline ganador de cada modelo (ya reentrenado con todo el train) y su `ganador.json`. Se agrega la **logística benchmark original** (`LogisticRegression(class_weight="balanced", max_iter=1000)`, sin tuning) como modelo de referencia fijo. Se entrena aquí con el train, porque no tiene hiperparámetros, y solo se usa en el test.

In [ ]:
pipelines, ganadores = {}, {}
for clave in NOMBRES_CLF:
    pipelines[clave], ganadores[clave] = cargar_ganador_fase1(clave)

# Modelo de referencia fijo
pipelines = {CLAVE_BENCHMARK: crear_benchmark(SEED).fit(X_train, y_train), **pipelines}

pd.DataFrame({
    "Estrategia": [ESTRATEGIAS[g["estrategia"]] for g in ganadores.values()],
    "PR-AUC fase 1": [f"{g['pr_auc_media']:.3f} ± {g['pr_auc_de']:.3f}" for g in ganadores.values()],
    "Hiperparámetros": [formatear_hiperparametros(g["hiperparametros"]) for g in ganadores.values()],
    "Dispositivo": [g["dispositivo_entrenamiento"] for g in ganadores.values()],
}, index=pd.Index([NOMBRES_CLF[c] for c in ganadores], name="Modelo"))

### Validación Cruzada

No se reentrena ningún modelo: las métricas de CV de cada ganador se toman de su fase 1 (`fase1/<modelo>/resultados_cv.csv`, fila de la estrategia ganadora). Ya se calcularon con los **mismos 5 folds** del train, así que son directamente comparables entre sí.

- **Métricas:** PR-AUC, ROC-AUC y, con umbral 0.5 sobre `predict_proba`, precision, recall, F1 y accuracy.
- **Tabla:** media ± desviación, ordenada por PR-AUC, con la diferencia frente a la logística benchmark.

**Benchmark (no comparable):** las cifras de la logística benchmark son las de su validación cruzada en el EDA (`EDA.ipynb`, sección *Modelos Benchmark Iniciales*). Esa CV usó también 5 folds estratificados con `SEED`, pero sobre **todo el dataset** (train + test, 4810 filas) y no solo sobre el train, así que sus folds son distintos y su desviación no es comparable. Se incluye como referencia orientativa, y la columna Δ frente a ella debe leerse con esa cautela. La comparación pareada con el benchmark se hace en el test.

**Nota:** la PR-AUC de CV de los modelos con tuning es ligeramente optimista, porque sus hiperparámetros y su estrategia se eligieron con estos mismos folds; el benchmark no tiene ese sesgo. Por eso el test es la comparación final.

**Umbral 0.5:** las métricas con umbral 0.5 **no son comparables entre modelos con distinta estrategia de balanceo**. Un modelo sin balanceo tendrá probabilidades más bajas y un recall menor con 0.5 que uno con pesos, aunque ordene igual de bien. La comparación justa es la de PR-AUC y ROC-AUC; el umbral se optimiza en la fase 3.

In [ ]:
# Logística benchmark: CV del EDA (EDA.ipynb, todo el dataset). No comparable con la CV del train.
CV_BENCHMARK_EDA = {
    "PR-AUC media": 0.399, "PR-AUC DE": 0.011,
    "ROC-AUC media": 0.766, "ROC-AUC DE": 0.007,
    "Precision media": 0.320, "Precision DE": 0.013,
    "Recall media": 0.666, "Recall DE": 0.035,
    "F1 media": 0.432, "F1 DE": 0.019,
    "Accuracy media": 0.714, "Accuracy DE": 0.009,
}

tabla_cv = cargar_cv_fase1(ganadores, CV_BENCHMARK_EDA)
formatear_tabla_cv(tabla_cv).rename(index=NOMBRES_FASE2)

### Selección del Mejor Modelo

Entre los 9 ganadores de la fase 1 se elige el de **mayor PR-AUC media de CV**.

La logística benchmark no compite: es la referencia contra la que se mide la ganancia.

In [ ]:
clave_mejor, explicacion = elegir_mejor(tabla_cv.drop(index=CLAVE_BENCHMARK))

pd.Series({
    "Modelo elegido": NOMBRES_CLF[clave_mejor],
    "Estrategia": ESTRATEGIAS[ganadores[clave_mejor]["estrategia"]],
    "PR-AUC de CV": f"{tabla_cv.loc[clave_mejor, 'PR-AUC media']:.4f} ± {tabla_cv.loc[clave_mejor, 'PR-AUC DE']:.4f}",
    "Δ PR-AUC de CV vs benchmark": f"{tabla_cv.loc[clave_mejor, 'Δ PR-AUC vs benchmark']:+.4f}",
    "Criterio": explicacion,
}, name="Selección").to_frame()

### Test

Los 10 modelos (9 ganadores + benchmark), ya entrenados con todo el train, se evalúan **una única vez** en el test. Las métricas de clase usan el umbral 0.5, con la misma advertencia de comparabilidad de la sección anterior.

In [ ]:
probas_test = {clave: pipeline.predict_proba(X_test)[:, 1] for clave, pipeline in pipelines.items()}

tabla_test = evaluar_test_modelos(probas_test, y_test, umbral=UMBRAL)
tabla_test.rename(index=NOMBRES_FASE2).round(3)

**Curvas ROC y Precision-Recall**

Curvas de los 10 modelos en el test. La logística benchmark aparece resaltada (línea gruesa en color tinta); las logísticas van con línea discontinua, KNN y SVC con trazo y punto, y los modelos de árboles con línea continua o punteada.

In [ ]:
fig = graficar_curvas_test(probas_test, y_test)
plt.show()

**Intervalos bootstrap al 95%**

Con 962 adolescentes en el test (unos 157 deprimidos), la PR-AUC y la ROC-AUC tienen bastante variabilidad de muestreo. Para saber si las diferencias entre modelos son reales:

- **Réplicas:** se generan **1000 réplicas bootstrap** del test, remuestreando adolescentes con reemplazo.
- **Evaluación pareada:** en cada réplica se evalúan **todos los modelos sobre las mismas observaciones**.
- **Intervalos:** se obtiene el intervalo al 95% (percentiles 2.5 y 97.5) de la PR-AUC y la ROC-AUC de cada modelo, y también el de la **diferencia pareada frente al benchmark**.
- **Lectura:** si el intervalo de la diferencia **incluye el 0**, no hay evidencia de que el modelo sea distinto del benchmark. La columna `significativa` lo marca.

El intervalo de la diferencia es más informativo que comparar los intervalos individuales: como los modelos se evalúan sobre las mismas réplicas, buena parte de la variabilidad se cancela.

In [ ]:
tabla_bootstrap = intervalos_bootstrap(y_test, probas_test, n_replicas=1000, seed=SEED)

columnas_pr = ["PR-AUC", "PR-AUC IC 2.5%", "PR-AUC IC 97.5%",
               "Δ PR-AUC vs benchmark", "Δ PR-AUC IC 2.5%", "Δ PR-AUC IC 97.5%", "Δ PR-AUC significativa"]
tabla_bootstrap[columnas_pr].rename(index=NOMBRES_FASE2).round(3)

In [ ]:
columnas_roc = ["ROC-AUC", "ROC-AUC IC 2.5%", "ROC-AUC IC 97.5%",
                "Δ ROC-AUC vs benchmark", "Δ ROC-AUC IC 2.5%", "Δ ROC-AUC IC 97.5%", "Δ ROC-AUC significativa"]
tabla_bootstrap[columnas_roc].rename(index=NOMBRES_FASE2).round(3)

In [ ]:
fig = graficar_intervalos_bootstrap(tabla_bootstrap)
plt.show()

**Cómo leer estos resultados:**

- **Intervalos individuales:** si los intervalos de todos los modelos se solapan ampliamente y ningún intervalo de la diferencia excluye el 0, los modelos son **estadísticamente indistinguibles** en el test.
- **Diferencias pequeñas:** en ese caso, las diferencias de PR-AUC en las tablas se deben a la variabilidad de muestreo.
- **Elección del modelo:** en ese caso, el modelo con mayor PR-AUC de CV no es claramente mejor que los demás, y conviene tener en cuenta también su simplicidad e interpretabilidad.

## Fase 3: Optimización del Umbral

El umbral 0.5 no está optimizado, y su significado depende de la estrategia de balanceo de cada modelo. En esta fase se elige el umbral del modelo seleccionado **solo con el train**:

1. **Probabilidades:** se calculan probabilidades *out-of-fold* con `cross_val_predict` (mismos folds, `method="predict_proba"`). Cada adolescente del train recibe la probabilidad de un modelo que no lo vio al entrenar, así que son una estimación honesta de cómo se comporta el modelo con datos nuevos.
2. **Criterio principal (tamizaje):** la **mayor precision con recall ≥ 0.70**. Es decir, detectar al menos el 70% de los adolescentes deprimidos con la menor cantidad posible de falsas alarmas.
3. **Referencia:** el umbral que **maximiza el F1**.
4. **Test:** el umbral principal se aplica **una sola vez** al test y se compara con el umbral 0.5.

**El test nunca se usa para elegir el umbral.**

### Probabilidades Out-of-Fold

In [ ]:
proba_oof = probabilidades_oof(pipelines[clave_mejor], clave_mejor, X_train, y_train, cv)

pd.Series({
    "Probabilidad media out-of-fold": round(proba_oof.mean(), 3),
    "Prevalencia (train)": round(y_train.mean(), 3),
}, name=NOMBRES_CLF[clave_mejor]).to_frame()

### Elección del Umbral

In [ ]:
umbrales = elegir_umbrales(y_train, proba_oof, recall_minimo=RECALL_MINIMO)
pd.Series(umbrales, name="Valor").to_frame()

In [ ]:
# Métricas out-of-fold (train) con cada umbral
candidatos = {
    f"Umbral 0.5": UMBRAL,
    f"Umbral principal ({umbrales['umbral']:.3f})": umbrales["umbral"],
    f"Umbral máximo F1 ({umbrales['umbral_f1']:.3f})": umbrales["umbral_f1"],
}
pd.DataFrame({nombre: metricas_con_umbral(y_train, proba_oof, u) for nombre, u in candidatos.items()}).T.round(3)

In [ ]:
fig = graficar_umbral(y_train, proba_oof, umbrales)
plt.show()

**Cómo leer la gráfica:**

- **Curvas:** al subir el umbral, el modelo marca menos positivos, así que el recall baja y la precision tiende a subir.
- **Umbral principal (línea continua):** es el umbral más exigente que todavía detecta al menos el 70% de los casos (línea horizontal punteada); por la derecha de ese punto, el recall cae por debajo de 0.70.
- **Umbral máximo F1 (línea discontinua):** sirve como referencia de equilibrio entre precision y recall.

### Aplicación al Test

El umbral principal, fijado con el train, se aplica **una sola vez** al test y se compara con el umbral 0.5. La PR-AUC y la ROC-AUC son iguales en ambas filas, porque no dependen del umbral.

In [ ]:
proba_test_mejor = probas_test[clave_mejor]

metricas_test_final = pd.DataFrame({
    "Umbral 0.5": metricas_con_umbral(y_test, proba_test_mejor, UMBRAL),
    "Umbral elegido": metricas_con_umbral(y_test, proba_test_mejor, umbrales["umbral"]),
}).T
metricas_test_final.insert(0, "Umbral", [UMBRAL, umbrales["umbral"]])
metricas_test_final["Alertas (%)"] = [
    (proba_test_mejor >= UMBRAL).mean() * 100,
    (proba_test_mejor >= umbrales["umbral"]).mean() * 100,
]
metricas_test_final.index.name = NOMBRES_CLF[clave_mejor]
metricas_test_final.round(3)

In [ ]:
predicciones = {
    f"Umbral 0.5": (proba_test_mejor >= UMBRAL).astype(int),
    f"Umbral elegido ({umbrales['umbral']:.3f})": (proba_test_mejor >= umbrales["umbral"]).astype(int),
}
fig = graficar_matrices(y_test, predicciones, f"{NOMBRES_CLF[clave_mejor]}: matrices de confusión en test")
plt.show()

**Cómo leer estos resultados:**

- **Recall en test:** se espera que quede cerca de 0.70. Un valor algo menor es normal, porque el umbral se eligió con otra muestra, pero uno muy por debajo indicaría que el umbral no generaliza bien.
- **% de alertas:** es la proporción de adolescentes que el modelo derivaría a una evaluación, e indica la carga de trabajo que generaría usarlo para tamizaje.
- **Matrices de confusión:** muestran cuántos casos detectados se ganan o se pierden frente al umbral 0.5 y a cambio de cuántas falsas alarmas.

### Persistencia del Modelo Final

Se guarda en `data/processed/modelado/final/` (excluido de git):

- `modelo_final.joblib`: el Pipeline del modelo elegido, entrenado con todo el train y configurado para CPU. Incluye el preprocesamiento y el modelo; si la estrategia ganadora es SMOTENC, también el paso de SMOTENC, que solo actúa al entrenar y se omite al predecir. Recibe un DataFrame con las columnas originales (`PREDICTORES`).
- `metricas_test.csv`: las métricas de test del modelo final con el umbral 0.5 y con el umbral elegido.
- `metadatos.json`: modelo, estrategia de balanceo (nativo, pesos o SMOTENC), hiperparámetros, umbral elegido y su criterio, umbral de máximo F1, SEED, predictores, dispositivo (CPU/GPU) de cada modelo y versiones de las librerías.

Para usarlo: `modelo, metadatos = cargar_modelo_final()` y `prediccion = modelo.predict_proba(df[PREDICTORES])[:, 1] >= metadatos["umbral"]`.

In [ ]:
metadatos = guardar_final(
    pipelines[clave_mejor], clave_mejor, ganadores[clave_mejor], umbrales, metricas_test_final, explicacion,
)
metadatos

In [ ]:
# Verificación: el modelo recargado desde disco da las mismas probabilidades
modelo_cargado, metadatos_cargados = cargar_modelo_final()
proba_disco = modelo_cargado.predict_proba(X_test[PREDICTORES])[:, 1]

pd.Series({
    "Mismas probabilidades": np.allclose(proba_disco, proba_test_mejor),
    "Umbral guardado": metadatos_cargados["umbral"],
}, name="Verificación").to_frame()

### Conclusiones

*Completar tras ejecutar el notebook.* Puntos a cubrir:

- **Modelo elegido:** cuál es, con qué estrategia de desbalance y por qué. La explicación se muestra en la celda de selección y queda en `metadatos.json`.
- **Ganancia frente al benchmark:** cuánto supera a la logística benchmark en PR-AUC de CV y de test, y si esa diferencia es real según el intervalo bootstrap de la diferencia pareada (columna `significativa`).
- **Diferencias entre modelos:** si son mayores que la desviación de CV y que la variabilidad del test. Si no lo son, el techo de desempeño lo ponen los 6 predictores y no la elección del modelo.
- **Umbral:** el umbral elegido, su precision y recall *out-of-fold*, y cómo se comportó en el test frente al 0.5 (recall, precision y % de alertas).
- **Limitaciones:** el umbral se eligió con un modelo entrenado en el 80% del train (cada fold) y se aplica al modelo entrenado con el 100%. Además, un recall de 0.70 implica que se escapan 3 de cada 10 casos.